# ANALIZA DANYCH Z ZEGARKA APPLE WATCH


## Podstawowe informacje o pliku:

* Plik jest pobierany z aplikacji Zdrowie (Apple Health) na telefonie
* format: XML
* struktura: hierarchiczna, z wieloma różnymi typami rekordów (Record, Correlation, Workout, ActivitySummary, ClinicalRecord, Audiogram, VisionPrescription)
* zawiera dane dotyczące zdrowia, aktywności, snu, tętna, kroków, itp.
* jest dość duży ponad 700 mb





Me
*   HKCharacteristicTypeIdentifierDateOfBirth
*   HKCharacteristicTypeIdentifierBiologicalSex
*   HKCharacteristicTypeIdentifierBloodType
*   HKCharacteristicTypeIdentifierFitzpatrickSkinType
*   HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse

Record
*   type
*   unit
*   value
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   ~~endDate~~~~~~

Workout ((MetadataEntry|WorkoutEvent|WorkoutRoute|WorkoutStatistics)*)
Workout
*   workoutActivityType
*   duration
*   durationUnit
*   totalDistance
*   totalDistanceUnit
*   totalEnergyBurned
*   totalEnergyBurnedUnit
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   endDate

WorkoutActivity
*   uuid
*   startDate
*   endDate
*   duration
*   durationUnit

WorkoutEvent
*   type
*   date
*   duration
*   durationUnit

WorkoutStatistics
*   type
*   startDate
*   endDate
*   average
*   minimum
*   maximum
*   sum
*   unit

WorkoutRoute
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   endDate


ActivitySummary
*   dateComponents
*   activeEnergyBurned
*   activeEnergyBurnedGoal
*   activeEnergyBurnedUnit
*   appleMoveTime
*   appleMoveTimeGoal
*   appleExerciseTime
*   appleExerciseTimeGoal
*   appleStandHours
*   appleStandHoursGoal

MetadataEntry
  key   
  value 

<!-- Note: Heart Rate Variability records captured by Apple Watch may include an associated list of instantaneous beats-per-minute readings. -->
<!ELEMENT HeartRateVariabilityMetadataList (InstantaneousBeatsPerMinute*)>
<!ELEMENT InstantaneousBeatsPerMinute EMPTY>
<!ATTLIST InstantaneousBeatsPerMinute
  bpm  
  time 
>

## TO DO:
1. [x] rozgryźć strukturę pliku xml - pól w nim zawartych - co dokładnie w nim jest
2. [x] sprawdzić pomiar EKG i możliwość ściągnięcia danych
3. [x] napisać kod pythona do wczytania tych danych
4. [x] zastanowić się nad programem ćwiczeń dla osób o różnej kondycji, z pomiarem zegarkiem

In [4]:
import os
import streamlit as st
import time
from lxml import etree as ET
import pandas as pd


In [5]:
path = "eksport.xml"

In [6]:
def discover_xml_structure(file_path):
    # Słownik: Klucz = Nazwa Tagu, Wartość = Zbiór atrybutów 'type' lub 'workoutActivityType'
    found_structures = {}

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        tag_name = elem.tag

        # Jeśli pierwszy raz widzimy ten tag, dodaj go do słownika
        if tag_name not in found_structures:
            found_structures[tag_name] = set()


        # Sprawdzamy, czy ten tag ma jakieś "podtypy" (np. HKQuantity...)
        sub_type = elem.get("type") or elem.get("workoutActivityType") or "Brak podtypu"

        found_structures[tag_name].add(sub_type)

        # Czyścimy RAM
        elem.clear()

    return found_structures

moje_dane = discover_xml_structure(path)

print("\n ODKRYTE ELEMENTY")
for tag, sub_types in moje_dane.items():
    print(f"\n TAG: <{tag}>")
    print(f"   Liczba różnych typów: {len(sub_types)}")
    for st in list(sub_types):
        print(f"     - {st}")


 ODKRYTE ELEMENTY

 TAG: <ExportDate>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Me>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Record>
   Liczba różnych typów: 41
     - HKQuantityTypeIdentifierAppleSleepingWristTemperature
     - HKQuantityTypeIdentifierHeartRateRecoveryOneMinute
     - HKQuantityTypeIdentifierWalkingHeartRateAverage
     - HKQuantityTypeIdentifierAppleWalkingSteadiness
     - HKQuantityTypeIdentifierHeartRate
     - HKCategoryTypeIdentifierAppleStandHour
     - HKQuantityTypeIdentifierTimeInDaylight
     - HKQuantityTypeIdentifierActiveEnergyBurned
     - HKQuantityTypeIdentifierBodyFatPercentage
     - HKQuantityTypeIdentifierSixMinuteWalkTestDistance
     - HKQuantityTypeIdentifierWalkingSpeed
     - HKCategoryTypeIdentifierAudioExposureEvent
     - HKQuantityTypeIdentifierBasalEnergyBurned
     - HKQuantityTypeIdentifierWalkingAsymmetryPercentage
     - HKQuantityTypeIdentifierHeartRateVariabilitySDNN
     - HKQuantityTypeIdentifierOxygen

In [7]:
important_tags = {
    'Me',"Record", "Workout", "WorkoutEvent", "WorkoutActivity",
    "WorkoutStatistics", "ActivitySummary", "InstantaneousBeatsPerMinute"
}


In [31]:

diff_types = set()

print("--- PRZEGLĄD STRUKTURY REKORDÓW ---")
diff_tags = set()
# Iterujemy po pliku
for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag in important_tags:
        typ = elem.get("type") or elem.tag or elem.get("workoutActivityType") or "Brak podtypu"

        if typ not in diff_types:
            print(f"\n TYP: {typ}")
            print(f"   Atrybuty: {elem.attrib}")
            diff_tags.add(typ)

            children = list(elem)
            if children:
                for child in children:
                    print(f"   └── Pod-element: <{child.tag}> atrybuty: {child.attrib}")

            diff_types.add(typ)



        elem.clear()
print("Lista różnych typów")
print(diff_tags)

--- PRZEGLĄD STRUKTURY REKORDÓW ---

 TYP: Me
   Atrybuty: {'HKCharacteristicTypeIdentifierDateOfBirth': '2006-03-02', 'HKCharacteristicTypeIdentifierBiologicalSex': 'HKBiologicalSexFemale', 'HKCharacteristicTypeIdentifierBloodType': 'HKBloodTypeNotSet', 'HKCharacteristicTypeIdentifierFitzpatrickSkinType': 'HKFitzpatrickSkinTypeNotSet', 'HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse': 'Brak'}

 TYP: HKQuantityTypeIdentifierHeight
   Atrybuty: {'type': 'HKQuantityTypeIdentifierHeight', 'sourceName': 'FitnessOnline', 'sourceVersion': '206', 'unit': 'cm', 'creationDate': '2025-08-12 11:58:06 +0100', 'startDate': '2025-08-12 11:58:05 +0100', 'endDate': '2025-08-12 11:58:05 +0100', 'value': '163'}

 TYP: HKQuantityTypeIdentifierBodyMass
   Atrybuty: {'type': 'HKQuantityTypeIdentifierBodyMass', 'sourceName': 'Zdrowie', 'sourceVersion': '14.7.1', 'unit': 'kg', 'creationDate': '2021-09-06 10:46:43 +0100', 'startDate': '2021-09-06 10:46:00 +0100', 'endDate': '2021-09-06 10:46:00 +01

Wszystkie typy danych w moim pliku:
{'HKQuantityTypeIdentifierAppleSleepingWristTemperature', 'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute', 'Me', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKWorkoutEventTypeResume', 'HKQuantityTypeIdentifierAppleWalkingSteadiness', 'HKQuantityTypeIdentifierHeartRate', 'HKCategoryTypeIdentifierAppleStandHour', 'HKQuantityTypeIdentifierTimeInDaylight', 'HKQuantityTypeIdentifierActiveEnergyBurned', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierSixMinuteWalkTestDistance', 'ActivitySummary', 'HKQuantityTypeIdentifierWalkingSpeed', 'HKCategoryTypeIdentifierAudioExposureEvent', 'Workout', 'HKWorkoutEventTypePause', 'HKQuantityTypeIdentifierBasalEnergyBurned', 'HKQuantityTypeIdentifierWalkingAsymmetryPercentage', 'HKWorkoutEventTypeSegment', 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKQuantityTypeIdentifierRestingHeartRate', 'HKDataTypeSleepDurationGoal', 'InstantaneousBeatsPerMinute', 'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage', 'HKQuantityTypeIdentifierRespiratoryRate', 'HKQuantityTypeIdentifierAppleStandTime', 'HKQuantityTypeIdentifierPhysicalEffort', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierFlightsClimbed', 'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKCategoryTypeIdentifierHighHeartRateEvent', 'HKQuantityTypeIdentifierVO2Max', 'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent', 'HKCategoryTypeIdentifierMenstrualFlow', 'HKQuantityTypeIdentifierStepCount', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierEnvironmentalAudioExposure', 'HKQuantityTypeIdentifierDistanceWalkingRunning', 'HKQuantityTypeIdentifierWaistCircumference', 'HKQuantityTypeIdentifierStairAscentSpeed', 'HKQuantityTypeIdentifierBodyMass', 'HKQuantityTypeIdentifierHeight', 'HKQuantityTypeIdentifierDistanceCycling', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKQuantityTypeIdentifierWalkingStepLength', 'HKWorkoutEventTypeMarker'}

In [9]:
#Stworzenie dataframe z wybranym typem danych

short_names = {
    # --- CIAŁO / PODSTAWOWE ---
    'HKQuantityTypeIdentifierHeight': 'height',
    'HKQuantityTypeIdentifierBodyMass': 'weight',
    'HKQuantityTypeIdentifierWaistCircumference': 'waist',
    'HKQuantityTypeIdentifierBodyFatPercentage': 'fat_pct',

    # --- SERCE (HEART) ---
    'HKQuantityTypeIdentifierHeartRate': 'hr',
    'HKQuantityTypeIdentifierRestingHeartRate': 'rhr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage': 'hr_walk_avg',
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute': 'hrr_1min',
    'HKCategoryTypeIdentifierHighHeartRateEvent': 'hr_high_event',

    # --- AKTYWNOŚĆ / ENERGIA ---
    'HKQuantityTypeIdentifierStepCount': 'steps',
    'HKQuantityTypeIdentifierDistanceWalkingRunning': 'dist_walk',
    'HKQuantityTypeIdentifierDistanceCycling': 'dist_cycle',
    'HKQuantityTypeIdentifierBasalEnergyBurned': 'kcal_basal',
    'HKQuantityTypeIdentifierActiveEnergyBurned': 'kcal_active',
    'HKQuantityTypeIdentifierFlightsClimbed': 'flights',
    'HKQuantityTypeIdentifierAppleExerciseTime': 'exercise_min',
    'HKQuantityTypeIdentifierAppleStandTime': 'stand_min',
    'HKCategoryTypeIdentifierAppleStandHour': 'stand_hr',
    'HKQuantityTypeIdentifierPhysicalEffort': 'effort',
    'HKQuantityTypeIdentifierVO2Max': 'vo2max',

    # --- MOBILNOŚĆ / CHÓD ---
    'HKQuantityTypeIdentifierWalkingSpeed': 'walk_speed',
    'HKQuantityTypeIdentifierWalkingStepLength': 'walk_step_len',
    'HKQuantityTypeIdentifierWalkingAsymmetryPercentage': 'walk_asym',
    'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage': 'walk_dbl_supp',
    'HKQuantityTypeIdentifierStairAscentSpeed': 'stair_up_speed',
    'HKQuantityTypeIdentifierStairDescentSpeed': 'stair_down_speed',
    'HKQuantityTypeIdentifierAppleWalkingSteadiness': 'walk_steadiness',
    'HKQuantityTypeIdentifierSixMinuteWalkTestDistance': 'walk_6min',

    # --- ODDECH / POZIOMY ---
    'HKQuantityTypeIdentifierOxygenSaturation': 'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate': 'resp_rate',

    # --- SEN ---
    'HKCategoryTypeIdentifierSleepAnalysis': 'sleep',
    'HKDataTypeSleepDurationGoal': 'sleep_goal',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',

    # --- ŚRODOWISKO / DŹWIĘK ---
    'HKQuantityTypeIdentifierEnvironmentalAudioExposure': 'audio_env',
    'HKQuantityTypeIdentifierHeadphoneAudioExposure': 'audio_hp',
    'HKQuantityTypeIdentifierEnvironmentalSoundReduction': 'sound_red',
    'HKCategoryTypeIdentifierAudioExposureEvent': 'audio_event',
    'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent': 'audio_hp_event',
    'HKQuantityTypeIdentifierTimeInDaylight': 'daylight',

    # --- INNE ---
    'HKCategoryTypeIdentifierMenstrualFlow': 'menstr'
}


In [10]:


def load_all_health_data(file_path, mapping):
    """
    Funkcja zwraca słownik DataFrameów z krótkimi nazwami, żeby szybciej wpisywać.
    """
    if file_path is None:
        return "Zła ścieżka pliku, spróbuj ponownie"
    if mapping is None:
        mapping = types_list # jeśli nie mamy krótkich nazw używamy pobranych długich

    # 1. Tworzymy puste listy dla każdego krótkiego skrótu ze słownika
    data_buckets = {short_name: [] for short_name in mapping.values()}

    # 2. Startujemy iterparse
    # event="end" oznacza, że reagujemy, gdy zamknie się tag </Record>
    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        if elem.tag == "Record" or elem.tag == "Workout" or :
            long_type = elem.get("type")

            # 3. Jeśli ten typ jest w naszym słowniku krótkich nazw
            if long_type in mapping:
                short_name = mapping[long_type]

                # Wyciągamy dane do słownika (lekki format)
                record = {
                    'value': elem.get('value'),
                    'unit': elem.get('unit'),
                    'sourceName': elem.get('sourceName'),
                    'startDate': elem.get('startDate'),
                    'endDate': elem.get('endDate'),

                }

                for child in elem:
                    if child.tag == "MetadataEntry":
                        record[child.get('key')] = child.get('value')

                data_buckets[short_name].append(record)

            elem.clear()



        root = context.root

    # 4. Zamieniamy listy na DataFrames i naprawiamy typy
    final_dfs = {}
    print("📊 Konwertuję dane na tabele...")

    for short_name, records in data_buckets.items():
        if records: # Tworzymy DF tylko jeśli są dane
            df = pd.DataFrame(records)

            # Automatyczna konwersja na liczby i daty
            if 'value' in df.columns:
                df['value'] = pd.to_numeric(df['value'], errors='coerce')
            if 'startDate' in df.columns:
                df['startDate'] = pd.to_datetime(df['startDate'])
            if 'endDate' in df.columns:
                df['endDate'] = pd.to_datetime(df['endDate'])

            final_dfs[short_name] = df

    print("✅ Wszystkie dane wczytane!")
    return final_dfs



hd = load_all_health_data("eksport.xml", short_names)


df = hd.get('hr')
df.head()
df.describe()
df_steps = hd.get('steps')

🚀 Rozpoczynam jednorazowy odczyt pliku... To potrwa ok. 30-60 sek.
📊 Konwertuję dane na tabele...
✅ Wszystkie dane wczytane!


In [11]:
print(df.head())
print(df.describe())
print(df_steps.head())

   value       unit sourceName                 startDate  \
0   72.0  count/min  Zepp Life 2021-12-17 00:00:00+01:00   
1   70.0  count/min  Zepp Life 2021-12-17 00:01:00+01:00   
2   70.0  count/min  Zepp Life 2021-12-17 00:02:00+01:00   
3   69.0  count/min  Zepp Life 2021-12-17 00:03:00+01:00   
4   70.0  count/min  Zepp Life 2021-12-17 00:04:00+01:00   

                    endDate HKMetadataKeyHeartRateMotionContext  \
0 2021-12-17 00:00:59+01:00                                 NaN   
1 2021-12-17 00:01:59+01:00                                 NaN   
2 2021-12-17 00:02:59+01:00                                 NaN   
3 2021-12-17 00:03:59+01:00                                 NaN   
4 2021-12-17 00:04:59+01:00                                 NaN   

  HKMetadataKeySyncVersion HKMetadataKeySyncIdentifier  
0                      NaN                         NaN  
1                      NaN                         NaN  
2                      NaN                         NaN  
3       

Czyścimy  dane - usuwamy kolumny w których wszystkie wartości sa nan

In [12]:
for name, df in hd.items():
    # axis=1 oznacza kolumny, how='all' tylko te całkiem puste
    hd[name] = df.dropna(axis=1, how='all')

In [13]:
for name in hd:
    hd[name].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype                    
---  ------      --------------  -----                    
 0   value       1 non-null      int64                    
 1   unit        1 non-null      object                   
 2   sourceName  1 non-null      object                   
 3   startDate   1 non-null      datetime64[ns, UTC+01:00]
 4   endDate     1 non-null      datetime64[ns, UTC+01:00]
dtypes: datetime64[ns, UTC+01:00](2), int64(1), object(2)
memory usage: 172.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype                    
---  ------            --------------  -----                    
 0   value             8 non-null      float64                  
 1   unit              8 non-null      object                   
 2   sourceName        8 non-null      obj

In [14]:
hd['hr'].info()
print(hd['hr'].describe())
print(hd['hr'].tail(20))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405380 entries, 0 to 405379
Data columns (total 8 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   value                                405380 non-null  float64                  
 1   unit                                 405380 non-null  object                   
 2   sourceName                           405380 non-null  object                   
 3   startDate                            405380 non-null  datetime64[ns, UTC+01:00]
 4   endDate                              405380 non-null  datetime64[ns, UTC+01:00]
 5   HKMetadataKeyHeartRateMotionContext  282907 non-null  object                   
 6   HKMetadataKeySyncVersion             347 non-null     object                   
 7   HKMetadataKeySyncIdentifier          347 non-null     object                   
dtypes: datetime64[ns, UTC+01:00](2), fl

In [15]:
df = hd['hr']


In [16]:
# Usuwamy konkretne kolumny techniczne, których nie potrzebujemy do analizy tętna
df = hd['hr'].drop(columns=['HKMetadataKeySyncVersion', 'HKMetadataKeySyncIdentifier'], errors='ignore')

In [17]:
print(df.columns)

Index(['value', 'unit', 'sourceName', 'startDate', 'endDate',
       'HKMetadataKeyHeartRateMotionContext'],
      dtype='object')


In [18]:



# Sprawdzamy, czy początek i koniec są takie same
# Wynikiem będzie True dla punktów (np. tętno) i False dla przedziałów (np. kroki)
df['is_point'] = df['startDate'] == df['endDate']

# Szybki podgląd: ile masz przedziałów, a ile punktów?
print(df['is_point'].value_counts())

is_point
True     282560
False    122820
Name: count, dtype: int64


In [19]:
# Usuwamy rekordy, które mają IDENTYCZNY start, koniec I wartość
df_clean = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

print(f"Usunięto {len(df) - len(df_clean)} idealnych duplikatów.")

Usunięto 5617 idealnych duplikatów.


In [20]:
df['midPoint'] = df['startDate'] + (df['endDate'] - df['startDate']) / 2

In [21]:
# Resampling dla kroków (sumujemy w oknie 1-minutowym)
df_steps_1min = df_steps.set_index('startDate')['value'].resample('1min').sum()

In [22]:
print(df.columns)

Index(['value', 'unit', 'sourceName', 'startDate', 'endDate',
       'HKMetadataKeyHeartRateMotionContext', 'is_point', 'midPoint'],
      dtype='object')


In [23]:
# 1. USUWANIE DUPLIKATÓW
# Czyścimy rekordy, które są identyczne (poczatek, koniec, wartość)
df = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

# Szukamy słowa "Apple" i ignorujemy to, co jest między wyrazami
df = df[df['sourceName'].str.contains('Apple', na=False)]


# 3. KONTROLA DATY (Inżynierskie BHP)
# Wywalamy rekordy, gdzie koniec jest przed początkiem (błędy systemowe)
df = df[df['endDate'] >= df['startDate']]

# 4. PRZYGOTOWANIE DO RESAMPLINGU
# Musimy ustawić datę jako indeks, żeby Pandas mógł "pogrupować" czas
df = df.set_index('startDate')

# 5. RESAMPLING (Ujednolicenie do 1 minuty)
# Zamiast 400k nieregularnych punktów, robimy średnią dla każdej minuty
df_hr_1min = df['value'].resample('1min').mean()

# 6. USUNIĘCIE "DZIUR" (Opcjonalnie)
# Jeśli przez godzinę nie miałaś zegarka, resampling stworzy tam NaN. Usuwamy je.
df_hr_1min = df_hr_1min.dropna()

print(f"Sukces! Twoje dane tętna są teraz jednolite. Zostało {len(df_hr_1min)} czystych minut.")

Sukces! Twoje dane tętna są teraz jednolite. Zostało 170294 czystych minut.


In [27]:
display(df.tail(100))

,value,unit,sourceName,endDate,HKMetadataKeyHeartRateMotionContext,is_point,midPoint
startDate,,,,,,,
2026-03-12 06:18:43+01:00,71.1812,count/min,Apple Watch (Lol),2026-03-12 06:18:43+01:00,0,True,2026-03-12 06:18:43+01:00
2026-03-12 06:23:49+01:00,68.0000,count/min,Apple Watch (Lol),2026-03-12 06:23:49+01:00,0,True,2026-03-12 06:23:49+01:00
2026-03-12 06:20:31+01:00,64.0000,count/min,Apple Watch (Lol),2026-03-12 06:20:31+01:00,0,True,2026-03-12 06:20:31+01:00
2026-03-12 06:23:56+01:00,66.0000,count/min,Apple Watch (Lol),2026-03-12 06:23:56+01:00,0,True,2026-03-12 06:23:56+01:00
2026-03-12 06:30:44+01:00,63.0000,count/min,Apple Watch (Lol),2026-03-12 06:30:44+01:00,0,True,2026-03-12 06:30:44+01:00
...,...,...,...,...,...,...,...
2026-03-12 11:35:13+01:00,78.0000,count/min,Apple Watch (Lol),2026-03-12 11:35:13+01:00,0,True,2026-03-12 11:35:13+01:00
2026-03-12 11:35:14+01:00,76.0000,count/min,Apple Watch (Lol),2026-03-12 11:35:14+01:00,0,True,2026-03-12 11:35:14+01:00
2026-03-12 11:35:15+01:00,75.0000,count/min,Apple Watch (Lol),2026-03-12 11:35:15+01:00,0,True,2026-03-12 11:35:15+01:00
